# Session 3 Evaluation: Thinking Before Answering

Session 2 showed how sampling can send repeated runs down different
autoregressive paths. This notebook holds one small local model, one
arithmetic expression, and the sampling parameters fixed while changing
what intermediate work the model generates before its answer. We will compare:

1. a direct answer with Qwen's thinking mode disabled,
2. a prompted scratchpad with thinking mode still disabled,
3. Qwen's thinking mode with the original direct-answer prompt, and
4. Python code generated by the model and executed by Python.

Run the cells from top to bottom. Your exact outputs may differ with a
different Ollama or model build; that is why we measure instead of
assuming.

We will propose an explanation for why intermediate tokens might help, but
the eval only tells us whether they helped this model in this setup. One
small sample estimates behavior; it does not prove either a universal rule
or the cause of the result.

## Settings

The correct result is `310`: multiplication happens before addition, so
`110 + 100 * 2` becomes `110 + 200`.

All four conditions hold the Session 2 sampling parameters fixed at
`temperature=0.5`, `top_k=20`, and `top_p=0.75`. These settings reduce
variation without forcing greedy decoding.
Whenever multiple candidates survive, the runtime can still sample among
them. These are controlled settings, not a claim that they are optimal.

We evaluate two properties separately:

- **Math correct:** the condition-specific parser finds the answer `310`.
- **Format correct:** direct answers contain only an integer; scratch work
  ends with an exact `Final answer: N` line.

In [ ]:
from pathlib import Path

from helper_functions import (
    evaluate_arithmetic_samples,
    evaluate_model,
    generate_programs,
    generate_text_samples,
    reset_output_dir,
    run_in_python_container,
    show_summary_table,
    show_text_samples,
    show_text_summary_table,
    summarize_results,
    summarize_text_results,
)

MODEL = "qwen3.5:2b"
EXPRESSION = "110 + 100 * 2"
EXPECTED_ANSWER = 310
N_SAMPLES = 10
SYSTEM_PROMPT = ""
SAMPLING_TEMPERATURE = 0.5
SAMPLING_TOP_K = 20
SAMPLING_TOP_P = 0.75

## Experiment 1: Answer Directly, with Thinking Mode Off

The prompt requests the answer without first generating a textual
scratchpad. We use attempts 1–10 as
seeds and record Ollama's aggregate generated-token count and end-to-end
duration alongside correctness.

In [ ]:
DIRECT_PROMPT = (
    "Respond with only the integer answer to the following arithmetic "
    f"expression: {EXPRESSION}"
)

direct_samples = generate_text_samples(
    model_name=MODEL,
    condition="Direct, think=False",
    count=N_SAMPLES,
    prompt=DIRECT_PROMPT,
    system_prompt=SYSTEM_PROMPT,
    think=False,
    max_generated_tokens=1024,
    temperature=SAMPLING_TEMPERATURE,
    top_k=SAMPLING_TOP_K,
    top_p=SAMPLING_TOP_P,
)
direct_results = evaluate_arithmetic_samples(
    direct_samples, EXPECTED_ANSWER, answer_format="integer_only"
)
assert all(not sample["thinking"] for sample in direct_samples)

show_text_samples(direct_results, limit=10)
show_text_summary_table(
    {"Direct, thinking mode off": summarize_text_results(direct_results)}
)

### Pause and Reflect: Likely Text Is Not Exact Arithmetic

For a fixed model and context, logits are deterministic scores for plausible
next tokens. The runtime's sampling step is stochastic. Why can lowering
that sampling variance make the same wrong answer appear more often? Would
`top_k=1` make the answer mathematically correct, or only repeat the model's
highest-scoring continuation?

Arithmetic requires one answer determined by formal rules. A high logit only
says that a token is a likely continuation under the model. Keep that mismatch
in mind as we give the model room to produce intermediate values.

## Experiment 2: Prompted Scratchpad, with Thinking Mode Off

This condition does **not** use Qwen's thinking mode. The ordinary,
visible response gets room to write intermediate values before its final
line. The 100-word instruction encourages brevity, making the final answer
more likely to fit within the output budget.

> **New term: output budget**
>
> An **output budget** is the maximum number of tokens the runtime allows a
> model to generate for one response. If the model uses the whole budget on
> intermediate steps, it may stop before returning its final answer.

In [ ]:
SCRATCHPAD_PROMPT = (
    "Work through this expression step by step before answering. Briefly "
    "state the relevant order-of-operations rule, evaluate one operation "
    "at a time, and show each intermediate expression. "
    "End with Final answer: <integer>. Do not write or run code.\n\n"
    f"Expression: {EXPRESSION}"
)

scratchpad_samples = generate_text_samples(
    model_name=MODEL,
    condition="Prompted scratchpad, think=False",
    count=N_SAMPLES,
    prompt=SCRATCHPAD_PROMPT,
    system_prompt=SYSTEM_PROMPT,
    think=False,
    max_generated_tokens=512,
    temperature=SAMPLING_TEMPERATURE,
    top_k=SAMPLING_TOP_K,
    top_p=SAMPLING_TOP_P,
)
scratchpad_results = evaluate_arithmetic_samples(
    scratchpad_samples, EXPECTED_ANSWER, answer_format="final_answer"
)
assert all(not sample["thinking"] for sample in scratchpad_samples)

show_text_samples(scratchpad_results, limit=3)
show_text_summary_table(
    {
        "Direct, thinking mode off": summarize_text_results(
            direct_results
        ),
        "Prompted scratchpad, thinking mode off": summarize_text_results(
            scratchpad_results
        ),
    }
)

## Experiment 3: Qwen's Thinking Mode

We now enable Qwen's thinking mode but restore the **exact original
direct-answer prompt**. We hold the prompt, seeds, other generation
conditions, and the same maximum of 1,024 generated tokens, making `think`
the intended difference. Ollama parses the model's marked intermediate
region and returns that **reasoning trace** in its separate `thinking`
field. This response structure is a runtime interface, not a second neural
channel inside the model.

The token count is Ollama's aggregate `eval_count`; it includes the
reasoning trace and final response. The `num_predict=1024` value is an
output limit, not a calibrated reasoning-effort control. Small models can
loop in their scratch work and
reach the limit without returning a final answer.

In [ ]:
thinking_mode_samples = generate_text_samples(
    model_name=MODEL,
    condition="Thinking mode, think=True",
    count=N_SAMPLES,
    prompt=DIRECT_PROMPT,
    system_prompt=SYSTEM_PROMPT,
    think=True,
    max_generated_tokens=1024,
    temperature=SAMPLING_TEMPERATURE,
    top_k=SAMPLING_TOP_K,
    top_p=SAMPLING_TOP_P,
)
thinking_mode_results = evaluate_arithmetic_samples(
    thinking_mode_samples, EXPECTED_ANSWER, answer_format="integer_only"
)
assert all(sample["thinking"] for sample in thinking_mode_samples)

show_text_samples(
    thinking_mode_results, limit=1, include_thinking=True
)

### Compare the Three Answering Conditions

The format check is condition-specific: direct and thinking-mode responses must be
only an integer, while the prompted scratchpad must end with `Final answer: N`.
Compare math accuracy, format compliance, tokens, and latency rather than
collapsing every goal into one number.

In [ ]:
answer_summaries = {
    "Direct, thinking mode off": summarize_text_results(direct_results),
    "Prompted scratchpad, thinking mode off": summarize_text_results(
        scratchpad_results
    ),
    "Direct prompt, thinking mode on": summarize_text_results(
        thinking_mode_results
    ),
}
show_text_summary_table(answer_summaries)

## Experiment 4: Let Python Do the Arithmetic

An LLM is good at producing candidate software. Python is good at executing
arithmetic deterministically. We now evaluate that division of labor. A
generated program passes only if it exits successfully and prints exactly
`310`, with at most the conventional final newline produced by `print`.

In [ ]:
PROGRAM_SYSTEM_PROMPT = (
    "You are a Python programmer who writes simple, well structured "
    "programs. When asked to write a program, respond with a complete "
    "Python program. Code only. Do not use eval. Do not respond with markdown."
)

PROGRAM_PROMPT = (
    f"Write a Python program that computes and prints: {EXPRESSION}"
)

OUTPUT_DIR = Path("generated-programs/session-3")
DOCKER_IMAGE = "python:3.12"

reset_output_dir(OUTPUT_DIR)
generate_programs(
    model_name=MODEL,
    count=N_SAMPLES,
    output_dir=OUTPUT_DIR,
    prompt=PROGRAM_PROMPT,
    system_prompt=PROGRAM_SYSTEM_PROMPT,
    think=False,
    max_generated_tokens=256,
    temperature=SAMPLING_TEMPERATURE,
    top_k=SAMPLING_TOP_K,
    top_p=SAMPLING_TOP_P,
)

In [ ]:
program_results = evaluate_model(
    model_name=MODEL,
    output_dir=OUTPUT_DIR,
    test_cases=[([], str(EXPECTED_ANSWER))],
    docker_image=DOCKER_IMAGE,
)
program_summary = {MODEL: summarize_results(program_results)}
show_summary_table(program_summary)

for row in program_results:
    status = "PASS" if row["passed"] else "FAIL"
    print(f"{status}: {row['script']}")
    if not row["passed"]:
        print(Path(row["script"]).read_text())
        print(row["tests"][0])

### Generated Programs and Deterministic Execution

In the July 2026 reference rerun, **10/10 generated programs passed**.

Pick one passing saved script and run the **same file** twice. Its source no
longer changes, so Python should produce the same result on both runs.
That does not make the whole pipeline deterministic: generation selected the
program, and untrusted generated code still needs isolation and validation.

In [ ]:
successful = next(
    (row for row in program_results if row["passed"]), None
)

if successful is None:
    print("No passing script was available to rerun.")
else:
    print("Selected passing source:\n")
    print(Path(successful["script"]).read_text())
    for run_number in (1, 2):
        completed = run_in_python_container(
            script_path=successful["script"],
            args=[],
            docker_image=DOCKER_IMAGE,
        )
        print(
            f"run {run_number}: return code={completed.returncode}, "
            f"stdout={completed.stdout.strip()!r}"
        )

## What to Take Away

- A directly generated answer is not the same thing as arithmetic execution.
- Low-variance sampling can repeat a wrong answer reliably; it does not turn
  likely next-token continuations into arithmetic rules.
- Tokens in a prompted scratchpad can become useful context for later tokens,
  even when Qwen's thinking mode is disabled.
- In this run, enabling thinking mode improved arithmetic accuracy, but it
  consumed more tokens and time and can itself fail.
- Generating a program moves deterministic work into a deterministic tool,
  while leaving generation, validation, and sandboxing as engineering work.